## 0. Setup

In [ ]:
#!/usr/bin/env python3
"""
Compare biospecimen source ontology terms between EpiATLAS and ENCODE datasets.

Uses ontology hierarchies (CL, UBERON, EFO, CLO) to assess semantic similarity
between biospecimen sources, going beyond exact string matching. This addresses
reviewer concerns about whether "distinct" biospecimen labels actually represent
biologically similar cell types (e.g., K562 vs Jurkat, H1 vs H9).

Approach:
  1. Load CL and UBERON ontologies locally via obonet -> nxontology (fast batch).
  2. For EFO and CLO terms, query the OLS4 REST API to retrieve ancestors.
  3. Build a unified NXOntology graph merging all sources.
  4. Compute pairwise Lin semantic similarity for all unique term pairs.
  5. Classify overlap into tiers: exact, high, moderate, low, distinct.

Dependencies:
    pip install obonet nxontology networkx pandas requests

Usage:
    # Prepare two DataFrames with an 'ontology_id' column containing
    # terms like 'CL:0000236', 'UBERON:0002107', 'EFO:0001187', etc.
"""
# pylint: disable=redefined-outer-name, too-many-branches, too-many-nested-blocks, too-many-lines, import-error, broad-exception-caught
import json
import logging
import time
from collections import defaultdict
from itertools import product
from pathlib import Path
from typing import Dict, Optional
from urllib.parse import quote_plus

import networkx as nx
import numpy as np
import obonet
import pandas as pd
import requests
from IPython.display import display
from nxontology import NXOntology
from nxontology.imports import multidigraph_to_digraph

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

Configuration

In [ ]:
project_path = Path.home() / "Projects/epiclass/output/paper"

OBO_CACHE_DIR = project_path / "data" / "obo"
OBO_CACHE_DIR.mkdir(exist_ok=True)

NXONTOLOGY_CACHE_PATH = project_path / "data" / "nxontology" / "nxontology_cache.json"
NXONTOLOGY_CACHE_PATH.parent.mkdir(exist_ok=True)

In [ ]:
# OBO ontology URLs (these have .obo files available)
OBO_URLS = {
    "CL": "http://purl.obolibrary.org/obo/cl.obo",
    "UBERON": "http://purl.obolibrary.org/obo/uberon.obo",
}

# OLS4 base URL for REST API queries
OLS4_BASE = "https://www.ebi.ac.uk/ols4/api"

# Similarity classification thresholds (Lin similarity)
THRESHOLDS = {
    "high": 0.7,  # Biologically very similar
    "moderate": 0.4,  # Same broad lineage
    "low": 0.2,  # Distantly related
    # Below "low" -> "distinct"
}

# Rate limiting for OLS4 API
OLS4_DELAY_SECONDS = 0.15  # ~6-7 requests/sec, well within limits

# EPIATLAS EBI REST API for missing EpiATLAS terms
EPIRR_API_BASE = "https://www.ebi.ac.uk/epirr/api/v1/epigenome"
EPIRR_DELAY_SECONDS = 0.1  # rate-limit courtesy

BIOSPECIMEN_ID_COL = "biospecimen_ontology_id"

## 1. Load/save OBO ontologies/nxgraph

In [ ]:
def load_obo_ontologies(
    obo_urls: dict[str, str],
    cache_dir: Optional[Path] = OBO_CACHE_DIR,
) -> nx.MultiDiGraph:
    """
    Load CL and UBERON from .obo files into a single merged MultiDiGraph.

    obonet produces edges child -> parent (is_a direction).
    All nodes from both ontologies are merged into one graph so that
    cross-ontology is_a links (e.g., CL terms referencing UBERON) are
    preserved.

    Parameters
    ----------
    obo_urls : dict
        Mapping of ontology prefix to OBO file URL or local path.
    cache_dir : Path, optional
        If provided, download .obo files here on first run and reuse them
        on subsequent runs instead of re-downloading from the network.

    Returns
    -------
    nx.MultiDiGraph
        Merged graph with edges keyed by relationship type.
    """
    if cache_dir is not None:
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)

    merged = nx.MultiDiGraph()

    for prefix, url_or_path in obo_urls.items():
        source = url_or_path

        # If cache_dir is set and the source looks like a URL, try the cache
        if cache_dir is not None and url_or_path.startswith(("http://", "https://")):
            # Derive a local filename from the URL
            filename = url_or_path.rsplit("/", 1)[-1]  # e.g. "cl.obo"
            cached_path = cache_dir / filename

            if cached_path.exists():
                log.info("Loading %s from cache: %s", prefix, cached_path)
                source = str(cached_path)
            else:
                log.info(
                    "Downloading %s from %s -> %s ...",
                    prefix,
                    url_or_path,
                    cached_path,
                )
                resp = requests.get(url_or_path, timeout=120)
                resp.raise_for_status()
                cached_path.write_bytes(resp.content)
                log.info("  Saved %s (%.1f MB)", cached_path, len(resp.content) / 1e6)
                source = str(cached_path)
        else:
            log.info("Loading %s from %s ...", prefix, source)

        graph = obonet.read_obo(source)
        log.info(
            "  %s: %d nodes, %d edges",
            prefix,
            len(graph),
            graph.number_of_edges(),
        )
        # Merge into combined graph
        merged.update(graph)

    log.info(
        "Merged graph: %d nodes, %d edges",
        len(merged),
        merged.number_of_edges(),
    )
    return merged

In [ ]:
def save_nxontology(nxo: NXOntology, path: Path) -> None:
    """Save a frozen NXOntology to a JSON (or .json.gz) file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    nxo.write_node_link_json(str(path))
    log.info("Saved NXOntology (%d nodes) to %s", nxo.n_nodes, path)


def load_nxontology(path: Path) -> NXOntology:
    """Load an NXOntology from a previously saved JSON (or .json.gz) file."""
    path = Path(path)
    nxo = NXOntology.read_node_link_json(str(path))
    nxo.freeze()
    log.info("Loaded NXOntology (%d nodes) from %s", nxo.n_nodes, path)
    return nxo

In [ ]:
load_obo_ontologies(obo_urls=OBO_URLS, cache_dir=OBO_CACHE_DIR)

## 1.5 Fetch EpiRR missing ontology curie

In [ ]:
def fetch_epirr_ontology_curie(
    accession: str,
    session: Optional[requests.Session] = None,
    delay: float = EPIRR_DELAY_SECONDS,
) -> Optional[str]:
    """
    Query the EpiRR API for a single accession and return the
    harmonized_sample_ontology_curie, or None on failure / empty value.
    """
    sess = session or requests.Session()
    url = f"{EPIRR_API_BASE}?accession={accession}"
    time.sleep(delay)
    try:
        resp = sess.get(url, headers={"accept": "application/json"}, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        curie = data.get("ihec_harmonized_metadata", {}).get(
            "harmonized_sample_ontology_curie", ""
        )
        return curie if curie else None
    except Exception as e:
        log.warning("EpiRR lookup failed for %s: %s", accession, e)
        return None


def fill_missing_ontology_ids(
    df: pd.DataFrame,
    ontology_col: str = "ontology_id",
    accession_col: str = "epirr_accession",
    delay: float = EPIRR_DELAY_SECONDS,
) -> pd.DataFrame:
    """
    For rows where `ontology_col` is missing, query the EpiRR API
    using the EpiRR accession to fill in the ontology curie.

    Parameters
    ----------
    df : pd.DataFrame
        Must have `accession_col`. `ontology_col` may be missing or
        partially filled.
    ontology_col : str
        Column to fill with ontology curies (e.g., 'CL:0011020').
    accession_col : str
        Column containing EpiRR accessions (e.g., 'IHECRE00004894').
    delay : float
        Seconds to wait between API requests.

    Returns
    -------
    pd.DataFrame
        Copy of df with `ontology_col` filled where possible.
        A new column `ontology_id_source` indicates 'original' vs 'epirr'.
    """
    df = df.copy()

    # Ensure the ontology column exists
    if ontology_col not in df.columns:
        df[ontology_col] = None

    missing_mask = df[ontology_col].isna() | (df[ontology_col] == "")
    n_missing = missing_mask.sum()

    if n_missing == 0:
        log.info("No missing ontology IDs, nothing to fetch.")
        df["ontology_id_source"] = "original"
        return df

    log.info(
        "Fetching ontology IDs for %d / %d samples from EpiRR ...",
        n_missing,
        len(df),
    )

    # Deduplicate accessions to minimize API calls
    missing_accessions = df.loc[missing_mask, accession_col].dropna().unique()
    log.info("  %d unique EpiRR accessions to query.", len(missing_accessions))

    session = requests.Session()
    accession_to_curie: dict[str, Optional[str]] = {}

    for i, acc in enumerate(missing_accessions):
        if (i + 1) % 50 == 0 or i == 0:
            log.info("  EpiRR query %d / %d: %s", i + 1, len(missing_accessions), acc)
        accession_to_curie[acc] = fetch_epirr_ontology_curie(
            acc,
            session=session,
            delay=delay,
        )

    # Fill in the results
    n_filled = 0
    df["ontology_id_source"] = "original"
    for idx in df.index[missing_mask]:
        acc = df.at[idx, accession_col]
        if pd.isna(acc):
            continue
        curie = accession_to_curie.get(acc)
        if curie:
            df.at[idx, ontology_col] = curie
            df.at[idx, "ontology_id_source"] = "epirr"
            n_filled += 1

    n_still_missing = missing_mask.sum() - n_filled
    log.info(
        "  Filled %d / %d missing IDs. Still missing: %d",
        n_filled,
        n_missing,
        n_still_missing,
    )

    return df

## 2. OLS4 API helpers for EFO, CLO, and fallback lookups

In [ ]:
class OLS4Client:
    """Minimal client for the EBI OLS4 REST API."""

    def __init__(self, base_url: str = OLS4_BASE, delay: float = OLS4_DELAY_SECONDS):
        self.base_url = base_url.rstrip("/")
        self.delay = delay
        self.session = requests.Session()
        self.session.headers["Accept"] = "application/json"
        self._ancestor_cache: dict[str, set[str]] = {}

    def _term_iri(self, obo_id: str) -> str:
        """Convert 'EFO:0001187' -> IRI for OLS4 API."""
        prefix, local = obo_id.split(":", 1)
        # Most OBO ontologies use underscore in IRIs
        local_underscore = f"{prefix}_{local}"
        if prefix == "EFO":
            return f"http://www.ebi.ac.uk/efo/{local_underscore}"

        return f"http://purl.obolibrary.org/obo/{local_underscore}"

    def _ontology_id(self, obo_id: str) -> str:
        """Map prefix to OLS4 ontology id (lowercase)."""
        prefix = obo_id.split(":")[0]
        return prefix.lower()

    def get_term_label(self, obo_id: str) -> Optional[str]:
        """Get the human-readable label for a term."""
        ontology = self._ontology_id(obo_id)
        iri = self._term_iri(obo_id)
        encoded_iri = quote_plus(quote_plus(iri))  # double-encode
        url = f"{self.base_url}/ontologies/{ontology}/terms/{encoded_iri}"
        try:
            time.sleep(self.delay)
            resp = self.session.get(url, timeout=30)
            if resp.status_code == 200:
                return resp.json().get("label")
        except Exception as e:
            log.debug("OLS4 label lookup failed for %s: %s", obo_id, e)
        return None

    def get_ancestors(self, obo_id: str) -> set[str]:
        """
        Get all ancestor term OBO IDs for a given term via OLS4 API.

        Returns a set of ancestor OBO IDs (e.g., {'CL:0000000', 'CL:0000003', ...}).
        Results are cached.
        """
        if obo_id in self._ancestor_cache:
            return self._ancestor_cache[obo_id]

        ontology = self._ontology_id(obo_id)
        iri = self._term_iri(obo_id)
        encoded_iri = quote_plus(quote_plus(iri))  # double-encode for OLS4
        url = f"{self.base_url}/ontologies/{ontology}/terms/{encoded_iri}/ancestors"

        ancestors = set()
        page_url = url + "?size=500"  # large page to minimize requests

        while page_url:
            time.sleep(self.delay)
            try:
                resp = self.session.get(page_url, timeout=30)
                if resp.status_code != 200:
                    log.warning(
                        "OLS4 ancestors request failed for %s: HTTP %d",
                        obo_id,
                        resp.status_code,
                    )
                    break

                data = resp.json()
                terms = data.get("_embedded", {}).get("terms", [])
                for term in terms:
                    anc_obo_id = term.get("obo_id")
                    if anc_obo_id:
                        ancestors.add(anc_obo_id)

                # Handle pagination
                next_link = data.get("_links", {}).get("next", {}).get("href")
                page_url = next_link if next_link else None

            except Exception as e:
                log.warning("OLS4 request error for %s: %s", obo_id, e)
                break

        self._ancestor_cache[obo_id] = ancestors
        return ancestors

## 3. Build unified NXOntology

In [ ]:
def parse_ontology_prefix(term_id: str) -> str:
    """Extract the ontology prefix from an OBO ID like 'CL:0000236'."""
    return term_id.split(":")[0]


def resolve_compound_terms(
    raw_terms: list[str],
    separator: str = "::",
) -> tuple[dict[str, list[str]], list[str]]:
    """
    Parse compound ontology IDs into their atomic components.

    Parameters
    ----------
    raw_terms : list of str
        Original ontology IDs, some possibly compound
        (e.g., 'UBERON:0006095::UBERON:0006096').
    separator : str
        Delimiter between atomic terms in compound IDs.

    Returns
    -------
    compound_map : dict
        {original_term: [atom1, atom2, ...]} for every input term.
        Single terms map to a one-element list.
    unique_atoms : list
        Sorted deduplicated list of all atomic terms (for ontology loading).
    """
    compound_map = {}
    all_atoms = set()
    for term in raw_terms:
        atoms = [t.strip() for t in term.split(separator) if t.strip()]
        compound_map[term] = atoms
        all_atoms.update(atoms)
    return compound_map, sorted(all_atoms)


def partition_terms_by_source(
    terms: list[str],
) -> dict[str, list[str]]:
    """Group terms by their ontology prefix."""
    grouped: dict[str, list[str]] = defaultdict(list)
    for t in terms:
        grouped[parse_ontology_prefix(t)].append(t)
    return dict(grouped)


def get_id_to_name(graph: nx.MultiDiGraph) -> dict[str, str]:
    """Build term ID -> human-readable name mapping from obonet graph."""
    return {node_id: data.get("name", "") for node_id, data in graph.nodes(data=True)}

In [ ]:
def build_nxontology_from_obo(merged_multidigraph: nx.MultiDiGraph) -> NXOntology:
    """
    Convert obonet MultiDiGraph -> NXOntology.

    nxontology expects edges from superterm -> subterm (reversed from obonet).
    multidigraph_to_digraph handles:
      - filtering to is_a edges only (default rel_types)
      - reversing edge direction
      - collapsing parallel edges

    We use reduce=False because merging CL + UBERON can introduce cycles
    via cross-ontology imports, and nx.transitive_reduction requires a DAG.
    Transitive reduction is an optimization, not a requirement for similarity.

    If cycles remain even after is_a filtering (rare but possible from
    cross-ontology imports), we break them by removing back-edges.
    """
    log.info("Converting MultiDiGraph to NXOntology DiGraph ...")
    digraph = multidigraph_to_digraph(merged_multidigraph, reduce=False)

    # Check for cycles and break them if present
    if not nx.is_directed_acyclic_graph(digraph):
        log.warning(
            "DiGraph contains cycles after is_a filtering. "
            "Breaking cycles by removing back-edges ..."
        )
        n_removed = 0
        # Find and remove edges that participate in cycles
        while not nx.is_directed_acyclic_graph(digraph):
            # Find one cycle
            cycle = nx.find_cycle(digraph)
            # Remove the last edge in the cycle (arbitrary but consistent)
            u, v = cycle[-1][:2]
            digraph.remove_edge(u, v)
            n_removed += 1
        log.warning("  Removed %d edges to break cycles.", n_removed)

    nxo = NXOntology(digraph)
    nxo.freeze()
    log.info("NXOntology: %d nodes", nxo.n_nodes)
    return nxo


def augment_nxontology_with_ols4(
    nxo_graph: nx.DiGraph,
    terms: list[str],
    ols4: OLS4Client,
) -> NXOntology:
    """
    For terms not in the local graph (EFO, CLO, NTR, etc.), fetch their
    ancestors from OLS4 and add edges into the graph.

    This creates a best-effort unified graph. EFO imports CL/UBERON terms,
    so many EFO ancestors will already exist as nodes.

    Parameters
    ----------
    nxo_graph : nx.DiGraph
        DiGraph from the local NXOntology (edges: superterm -> subterm).
    terms : list of str
        OBO IDs that need to be in the graph.
    ols4 : OLS4Client
        Client for API queries.

    Returns
    -------
    NXOntology
        New frozen NXOntology with additional nodes/edges.
    """
    # Work on a mutable copy
    graph = nxo_graph.copy()

    missing = [t for t in terms if t not in graph]
    if not missing:
        log.info("All terms already in local graph, no OLS4 augmentation needed.")
        nxo = NXOntology(graph)
        nxo.freeze()
        return nxo

    log.info(
        "%d / %d terms missing from local graph, querying OLS4 ...",
        len(missing),
        len(terms),
    )

    # NTR is a placeholder namespace for unmapped terms, no point querying OLS4
    SKIP_PREFIXES = {"NTR"}

    for i, term_id in enumerate(missing):
        prefix = term_id.split(":")[0]
        if prefix in SKIP_PREFIXES:
            log.info("  Skipping %s (no ontology available for %s)", term_id, prefix)
            graph.add_node(term_id)  # still add as isolated node
            continue

        if (i + 1) % 10 == 0 or i == 0:
            log.info("  OLS4 query %d / %d: %s", i + 1, len(missing), term_id)

        ancestors = ols4.get_ancestors(term_id)
        if not ancestors:
            log.warning("  No ancestors found for %s, adding as isolated node.", term_id)
            graph.add_node(term_id)
            continue

        # Ensure term node exists
        if term_id not in graph:
            graph.add_node(term_id)

        # Add ancestor nodes and edges (superterm -> subterm direction)
        # We only add direct parent links. For OLS4, the full ancestor set
        # is flat — we don't get the full DAG structure from /ancestors.
        # Instead, we add all ancestors as potential shared nodes, and
        # connect the term to each ancestor with an edge.
        #
        # For semantic similarity based on ancestor overlap (Jaccard, etc.)
        # this is sufficient. For IC-based metrics (Lin), the graph
        # structure matters more — but the local CL/UBERON graph already
        # has the DAG structure for any CL/UBERON ancestors.
        for anc_id in ancestors:
            if anc_id not in graph:
                graph.add_node(anc_id)
            # Only add edge if not already present
            if not graph.has_edge(anc_id, term_id):
                graph.add_edge(anc_id, term_id)

    nxo = NXOntology(graph)
    nxo.freeze()
    log.info("Augmented NXOntology: %d nodes", nxo.n_nodes)
    return nxo

## 4. Similarity computation

In [ ]:
def compute_pairwise_similarity(
    nxo: NXOntology,
    terms_a: list[str],
    terms_b: list[str],
    ic_metric: str = "intrinsic_ic_sanchez",
) -> pd.DataFrame:
    """
    Compute pairwise semantic similarity between two sets of ontology terms.

    For each pair (a, b), computes Lin similarity using intrinsic IC.
    Terms not found in the graph are flagged with NaN.

    Parameters
    ----------
    nxo : NXOntology
        Frozen ontology graph.
    terms_a, terms_b : list of str
        Unique ontology term IDs.
    ic_metric : str
        Information content metric for nxontology.

    Returns
    -------
    pd.DataFrame
        Columns: term_a, term_b, lin_similarity, mica (most informative
        common ancestor), mica_name.
    """
    results = []
    total = len(terms_a) * len(terms_b)
    graph_nodes = set(nxo.graph.nodes)

    log.info(
        "Computing %d pairwise similarities (%d x %d) ...",
        total,
        len(terms_a),
        len(terms_b),
    )

    for i, (a, b) in enumerate(product(terms_a, terms_b)):
        if (i + 1) % 500 == 0:
            log.info("  Progress: %d / %d", i + 1, total)

        if a == b:
            results.append(
                {
                    "term_a": a,
                    "term_b": b,
                    "lin_similarity": 1.0,
                    "mica": a,
                }
            )
            continue

        if a not in graph_nodes or b not in graph_nodes:
            results.append(
                {
                    "term_a": a,
                    "term_b": b,
                    "lin_similarity": np.nan,
                    "mica": None,
                }
            )
            continue

        try:
            sim = nxo.similarity(a, b, ic_metric=ic_metric)
            results.append(
                {
                    "term_a": a,
                    "term_b": b,
                    "lin_similarity": sim.lin,
                    "mica": sim.mica,
                }
            )
        except Exception as e:
            # Nodes in disconnected components, etc.
            log.debug("Similarity failed for %s vs %s: %s", a, b, e)
            results.append(
                {
                    "term_a": a,
                    "term_b": b,
                    "lin_similarity": np.nan,
                    "mica": None,
                }
            )

    df = pd.DataFrame(results)
    log.info("Computed %d pairwise similarities.", len(df))
    return df

In [ ]:
def compound_pairwise_similarity(
    nxo: NXOntology,
    terms_a: list[str],
    terms_b: list[str],
    separator: str = "::",
    ic_metric: str = "intrinsic_ic_sanchez",
) -> pd.DataFrame:
    """
    Compute pairwise similarity between two sets of possibly-compound
    ontology terms. For compound terms, similarity is the MAX over all
    (atom_a, atom_b) pairs from the two compound terms.

    This avoids exploding DataFrames — the output has one row per
    (original_term_a, original_term_b) pair.
    """
    map_a, _ = resolve_compound_terms(terms_a, separator)
    map_b, _ = resolve_compound_terms(terms_b, separator)

    # Precompute all unique atom pairs we'll need
    graph_nodes = set(nxo.graph.nodes)
    atom_sim_cache: dict[tuple[str, str], float] = {}

    all_atom_pairs = set()
    for orig_a in terms_a:
        for orig_b in terms_b:
            for aa in map_a[orig_a]:
                for ab in map_b[orig_b]:
                    all_atom_pairs.add((aa, ab))

    log.info("Computing %d unique atom-pair similarities ...", len(all_atom_pairs))
    for i, (aa, ab) in enumerate(all_atom_pairs):
        if (i + 1) % 500 == 0:
            log.info("  Progress: %d / %d", i + 1, len(all_atom_pairs))

        if aa == ab:
            atom_sim_cache[(aa, ab)] = 1.0
            continue
        if aa not in graph_nodes or ab not in graph_nodes:
            atom_sim_cache[(aa, ab)] = float("nan")
            continue
        try:
            sim = nxo.similarity(aa, ab, ic_metric=ic_metric)
            atom_sim_cache[(aa, ab)] = sim.lin
        except Exception:
            atom_sim_cache[(aa, ab)] = float("nan")

    # Aggregate: max similarity over all atom pairs
    results = []
    for orig_a in terms_a:
        for orig_b in terms_b:
            atoms_for_a = map_a[orig_a]
            atoms_for_b = map_b[orig_b]

            sims = [atom_sim_cache[(aa, ab)] for aa in atoms_for_a for ab in atoms_for_b]
            # Filter NaN, take max of valid values
            valid = [s for s in sims if not np.isnan(s)]
            best_sim = max(valid) if valid else float("nan")

            # Find which atom pair gave the best similarity
            best_pair = None
            if valid:
                for aa in atoms_for_a:
                    for ab in atoms_for_b:
                        if atom_sim_cache[(aa, ab)] == best_sim:
                            best_pair = (aa, ab)
                            break
                    if best_pair:
                        break

            results.append(
                {
                    "term_a": orig_a,
                    "term_b": orig_b,
                    "lin_similarity": best_sim,
                    "best_atom_a": best_pair[0] if best_pair else None,
                    "best_atom_b": best_pair[1] if best_pair else None,
                    "n_atoms_a": len(atoms_for_a),
                    "n_atoms_b": len(atoms_for_b),
                }
            )

    return pd.DataFrame(results)

In [ ]:
def compute_jaccard_ancestor_similarity(
    nxo: NXOntology,
    terms_a: list[str],
    terms_b: list[str],
) -> pd.DataFrame:
    """
    Alternative: Jaccard similarity based on ancestor set overlap.

    This is more robust to disconnected graph components and works
    even when IC-based metrics fail (e.g., for OLS4-augmented terms
    where the DAG structure is incomplete).
    """
    graph_nodes = set(nxo.graph.nodes)

    # Precompute ancestor sets
    ancestor_cache: dict[str, set[str]] = {}
    for term in set(terms_a) | set(terms_b):
        if term in graph_nodes:
            try:
                info = nxo.node_info(term)
                ancestor_cache[term] = info.ancestors
            except Exception:
                ancestor_cache[term] = set()
        else:
            ancestor_cache[term] = set()

    results = []
    for a, b in product(terms_a, terms_b):
        anc_a = ancestor_cache.get(a, set())
        anc_b = ancestor_cache.get(b, set())

        if a == b:
            jaccard = 1.0
        elif not anc_a and not anc_b:
            jaccard = np.nan
        else:
            union = anc_a | anc_b
            intersection = anc_a & anc_b
            jaccard = len(intersection) / len(union) if union else 0.0

        results.append(
            {
                "term_a": a,
                "term_b": b,
                "jaccard_similarity": jaccard,
            }
        )

    return pd.DataFrame(results)

## 5. Classify and summarize overlap

In [ ]:
def classify_overlap(
    sim_df: pd.DataFrame,
    col: str = "lin_similarity",
    thresholds: Optional[Dict] = None,
) -> pd.DataFrame:
    """Add an 'overlap_tier' column based on similarity thresholds."""
    if thresholds is None:
        thresholds = THRESHOLDS

    conditions = [
        sim_df[col] == 1.0,
        sim_df[col] >= thresholds["high"],
        sim_df[col] >= thresholds["moderate"],
        sim_df[col] >= thresholds["low"],
    ]
    choices = ["exact", "high", "moderate", "low"]
    sim_df = sim_df.copy()
    sim_df["overlap_tier"] = np.select(conditions, choices, default="distinct")
    # NaN -> "unmapped"
    sim_df.loc[sim_df[col].isna(), "overlap_tier"] = "unmapped"
    return sim_df


def summarize_best_matches(
    sim_df: pd.DataFrame,
    col: str = "lin_similarity",
) -> pd.DataFrame:
    """
    For each term_b (ENCODE), find the best-matching term_a (EpiATLAS).
    This answers: for each ENCODE biospecimen, how similar is the closest
    EpiATLAS biospecimen?
    """
    valid = sim_df.dropna(subset=[col])
    if valid.empty:
        return pd.DataFrame()

    idx = valid.groupby("term_b")[col].idxmax()
    best = valid.loc[idx].copy()
    best = best.sort_values(col, ascending=False)
    return best


def print_summary(classified_df: pd.DataFrame, col: str = "overlap_tier"):
    """Print a summary table of overlap tiers."""
    counts = classified_df[col].value_counts()
    total = len(classified_df)
    print("\n" + "=" * 60)
    print("BIOSPECIMEN SOURCE OVERLAP SUMMARY")
    print("=" * 60)
    for tier in ["exact", "high", "moderate", "low", "distinct", "unmapped"]:
        n = counts.get(tier, 0)
        pct = 100 * n / total if total else 0
        print(f"  {tier:>10s}: {n:5d}  ({pct:5.1f}%)")
    print(f"  {'TOTAL':>10s}: {total:5d}")
    print("=" * 60)

In [ ]:
def run_comparison(
    df_epiatlas: pd.DataFrame,
    df_encode: pd.DataFrame,
    ontology_col: str = "ontology_id",
    output_prefix: str = "biospecimen_comparison",
    cache_path: Optional[Path] = NXONTOLOGY_CACHE_PATH,
    rebuild_cache: bool = False,
) -> dict:
    """
    Full pipeline: load ontologies, compute similarity, classify overlap.

    Parameters
    ----------
    df_epiatlas : pd.DataFrame
        EpiATLAS samples. Must have `ontology_col`.
    df_encode : pd.DataFrame
        ENCODE samples. Must have `ontology_col`.
    ontology_col : str
        Column name containing ontology IDs (e.g., 'CL:0000236').
    output_prefix : str
        Prefix for output CSV files.
    cache_path : Path or None
        Where to save/load the built NXOntology graph. None disables caching.
        Supports .json and .json.gz extensions.
    rebuild_cache : bool
        If True, ignore existing cache and rebuild from scratch.

    Returns
    -------
    dict with keys:
        'pairwise': full pairwise similarity DataFrame
        'best_matches': best EpiATLAS match for each ENCODE term
        'summary': classified overlap tiers
    """
    # --- Extract unique terms ---
    terms_epiatlas = sorted(df_epiatlas[ontology_col].dropna().unique())
    terms_encode = sorted(df_encode[ontology_col].dropna().unique())
    all_terms = sorted(set(terms_epiatlas) | set(terms_encode))

    log.info(
        "Unique terms — EpiATLAS: %d, ENCODE: %d, total: %d",
        len(terms_epiatlas),
        len(terms_encode),
        len(all_terms),
    )

    by_prefix = partition_terms_by_source(all_terms)
    log.info("Terms by ontology prefix: %s", {k: len(v) for k, v in by_prefix.items()})

    # --- Try loading from cache ---
    nxo = None
    id_to_name = {}

    if cache_path and Path(cache_path).exists() and not rebuild_cache:
        log.info("Loading cached NXOntology from %s ...", cache_path)
        nxo = load_nxontology(cache_path)
        # Check that all our terms are present in the cached graph
        cached_nodes = set(nxo.graph.nodes)
        missing_from_cache = [t for t in all_terms if t not in cached_nodes]
        if missing_from_cache:
            log.warning(
                "%d terms not found in cached graph — rebuilding. Missing: %s%s",
                len(missing_from_cache),
                missing_from_cache[:5],
                " ..." if len(missing_from_cache) > 5 else "",
            )
            nxo = None  # force rebuild
        else:
            log.info("All %d terms found in cached graph.", len(all_terms))
            # Build id_to_name from cached graph node attributes
            id_to_name = {
                n: data.get("name", n) for n, data in nxo.graph.nodes(data=True)
            }

    # --- Build from scratch if no valid cache ---
    if nxo is None:
        # Load local OBO ontologies (CL, UBERON)
        merged_graph = load_obo_ontologies(OBO_URLS)
        id_to_name = get_id_to_name(merged_graph)

        # Build initial NXOntology from local data
        nxo_initial = build_nxontology_from_obo(merged_graph)

        # Check which terms need OLS4 augmentation
        local_nodes = set(nxo_initial.graph.nodes)
        needs_ols4 = [t for t in all_terms if t not in local_nodes]
        log.info(
            "%d / %d terms found in local graph, %d need OLS4 lookup.",
            len(all_terms) - len(needs_ols4),
            len(all_terms),
            len(needs_ols4),
        )

        # Augment with OLS4 if needed
        if needs_ols4:
            ols4 = OLS4Client()
            nxo = augment_nxontology_with_ols4(
                nxo_initial.graph,
                all_terms,
                ols4,
            )
        else:
            nxo = nxo_initial

        # Save to cache
        if cache_path:
            save_nxontology(nxo, cache_path)

    # --- Compute similarity ---
    # Use only unique terms for the pairwise matrix
    pairwise_df = compute_pairwise_similarity(nxo, terms_epiatlas, terms_encode)

    # Also compute Jaccard as a fallback metric
    jaccard_df = compute_jaccard_ancestor_similarity(nxo, terms_epiatlas, terms_encode)
    pairwise_df = pairwise_df.merge(jaccard_df, on=["term_a", "term_b"], how="left")

    # --- Add human-readable names ---
    pairwise_df["term_a_name"] = pairwise_df["term_a"].map(lambda x: id_to_name.get(x, x))
    pairwise_df["term_b_name"] = pairwise_df["term_b"].map(lambda x: id_to_name.get(x, x))
    pairwise_df["mica_name"] = pairwise_df["mica"].map(
        lambda x: id_to_name.get(x, x) if x else None
    )

    # --- Classify and summarize ---
    # Best match for each ENCODE term
    best_df = summarize_best_matches(pairwise_df)
    best_classified = classify_overlap(best_df)

    # Print summary
    print_summary(best_classified)

    # --- Save outputs ---
    pairwise_path = f"{output_prefix}_pairwise.csv"
    best_path = f"{output_prefix}_best_matches.csv"

    pairwise_df.to_csv(pairwise_path, index=False)
    best_classified.to_csv(best_path, index=False)
    log.info("Saved pairwise similarities to %s", pairwise_path)
    log.info("Saved best matches to %s", best_path)

    return {
        "pairwise": pairwise_df,
        "best_matches": best_classified,
        "nxo": nxo,
    }

## DEMO

In [ ]:
def make_demo_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create small demo DataFrames for testing."""
    epiatlas = pd.DataFrame(
        {
            "sample_id": [
                "EPIATLAS_001",
                "EPIATLAS_002",
                "EPIATLAS_003",
                "EPIATLAS_004",
                "EPIATLAS_005",
            ],
            "ontology_id": [
                "CL:0000236",  # B cell
                "CL:0000084",  # T cell
                "CL:0002322",  # embryonic stem cell
                "UBERON:0002107",  # liver
                "CL:0000775",  # neutrophil
            ],
        }
    )

    encode = pd.DataFrame(
        {
            "sample_id": [
                "ENCODE_001",
                "ENCODE_002",
                "ENCODE_003",
                "ENCODE_004",
                "ENCODE_005",
            ],
            "ontology_id": [
                "CL:0000624",  # CD4-positive T cell (related to T cell)
                "CL:0000037",  # hematopoietic stem cell
                "CL:0002322",  # embryonic stem cell (exact match)
                "UBERON:0002048",  # lung (different organ from liver)
                "CL:0000738",  # leukocyte (parent of neutrophil)
            ],
        }
    )

    return epiatlas, encode

In [ ]:
# if __name__ == "__main__":
#     # ------------------------------------------------------------------
#     # DEMO MODE: run with synthetic data to verify the pipeline works.
#     # Replace this block with your actual data loading.
#     # ------------------------------------------------------------------
#     log.info("Running in demo mode with synthetic data ...")
#     df_epi, df_enc = make_demo_data()

#     # To use your real data, replace the above with something like:
#     # df_epi = pd.read_csv("epiatlas_samples.csv")
#     # df_enc = pd.read_csv("encode_samples.csv")
#     # Make sure both have an 'ontology_id' column with values like 'CL:0000236'

#     results = run_comparison(
#         df_epi,
#         df_enc,
#         ontology_col="ontology_id",
#         output_prefix="biospecimen_comparison",
#     )

#     # Show best matches
#     best = results["best_matches"]
#     print("\nBest EpiATLAS match for each ENCODE biospecimen:")
#     print(
#         best[
#             [
#                 "term_b",
#                 "term_b_name",
#                 "term_a",
#                 "term_a_name",
#                 "lin_similarity",
#                 "jaccard_similarity",
#                 "mica_name",
#                 "overlap_tier",
#             ]
#         ].to_string(index=False)
#     )

## MAIN

In [ ]:
def exclude_ntr(df: pd.DataFrame, col: str = BIOSPECIMEN_ID_COL) -> pd.DataFrame:
    """Exclude rows where the ontology ID starts with 'NTR:'."""
    mask = df[col].str.startswith("NTR:")
    n_excluded = mask.sum()
    log.info("Excluding %d rows with NTR terms in %s.", n_excluded, col)
    return df[~mask].copy()

ENCODE data

In [ ]:
encode_data_path = (
    project_path
    / "data"
    / "metadata"
    / "encode"
    / "encode_full_metadata_2025-02_no_revoked.freeze1.csv.xz"
)

encode_df = pd.read_csv(encode_data_path, compression="xz", low_memory=False)
print(encode_df.shape)

encode_df = encode_df[encode_df["in_epiatlas"].astype(str) == "False"]
print(f"ENCODE: {encode_df.shape[0]} total files with no EpiAtlas overlap")

encode_df[BIOSPECIMEN_ID_COL] = encode_df["BIOSAMPLE_TYPE_term_id"]
initial_encode_N = encode_df.shape[0]

In [ ]:
encode_df[BIOSPECIMEN_ID_COL].str.split(":").str[0].value_counts(dropna=False)

In [ ]:
# The terms were prechecked, the missing terms are all primary_cell_NTR_000066*, skipping them
encode_df = encode_df.fillna({BIOSPECIMEN_ID_COL: "NTR:MISSING"})

EpiATLAS data

In [ ]:
epiatlas_data_path = (
    project_path
    / "data"
    / "metadata"
    / "epiatlas"
    / "hg38_2023-epiatlas-dfreeze-pospurge-nodup_filterCtl.json"
)
with open(epiatlas_data_path, encoding="utf8") as f:
    epiatlas_data = json.load(f)
epiatlas_df = pd.DataFrame.from_records(epiatlas_data["datasets"])
print(epiatlas_df.shape)
del epiatlas_data

epiatlas_df[BIOSPECIMEN_ID_COL] = epiatlas_df["harmonized_sample_ontology_curie"]

initial_epiatlas_N = epiatlas_df.shape[0]

In [ ]:
epiatlas_df[BIOSPECIMEN_ID_COL].str.split(":").str[0].value_counts(dropna=False)

In [ ]:
epiatlas_df[epiatlas_df[BIOSPECIMEN_ID_COL].str.contains("::")][
    BIOSPECIMEN_ID_COL
].value_counts(dropna=False)

In [ ]:
epiatlas_df = fill_missing_ontology_ids(
    df=epiatlas_df,
    ontology_col=BIOSPECIMEN_ID_COL,
    accession_col="EpiRR_no-v",
)

In [ ]:
epiatlas_df = exclude_ntr(epiatlas_df)
encode_df = exclude_ntr(encode_df)

In [ ]:
print(epiatlas_df[BIOSPECIMEN_ID_COL].isna().sum())
print(encode_df[BIOSPECIMEN_ID_COL].isna().sum())

In [ ]:
# Remaining data post cleaning (%)
for df in [encode_df, epiatlas_df]:
    n = df.shape[0]
    pct = 100 * n / initial_encode_N if df is encode_df else 100 * n / initial_epiatlas_N
    source = "ENCODE" if df is encode_df else "EpiATLAS"
    print(f"{source}: {n} samples remaining ({pct:.1f}% of original)")

In [ ]:
encode_df["has_epiatlas_biospecimen_overlap"] = encode_df[BIOSPECIMEN_ID_COL].isin(
    epiatlas_df[BIOSPECIMEN_ID_COL]
)
overlap_count = encode_df["has_epiatlas_biospecimen_overlap"].sum()
print(
    f"ENCODE samples with EpiATLAS biospecimen overlap: {overlap_count} / {encode_df.shape[0]} ({100 * overlap_count / encode_df.shape[0]:.1f}%)"
)

In [ ]:
# --- Extract unique terms (possibly compound) ---
raw_epiatlas = sorted(epiatlas_df[BIOSPECIMEN_ID_COL].dropna().unique())
raw_encode = sorted(encode_df[BIOSPECIMEN_ID_COL].dropna().unique())

# Resolve compound terms (e.g., 'UBERON:0006095::UBERON:0006096') into atoms
map_epiatlas, atoms_epiatlas = resolve_compound_terms(raw_epiatlas)
map_encode, atoms_encode = resolve_compound_terms(raw_encode)
all_atoms = sorted(set(atoms_epiatlas) | set(atoms_encode))

n_compound_a = sum(1 for v in map_epiatlas.values() if len(v) > 1)
n_compound_b = sum(1 for v in map_encode.values() if len(v) > 1)
log.info(
    "Unique terms — EpiATLAS: %d (%d compound), ENCODE: %d (%d compound), "
    "total atomic: %d",
    len(raw_epiatlas),
    n_compound_a,
    len(raw_encode),
    n_compound_b,
    len(all_atoms),
)

by_prefix = partition_terms_by_source(all_atoms)
log.info("Atomic terms by ontology prefix: %s", {k: len(v) for k, v in by_prefix.items()})

In [ ]:
# --- Try loading from cache ---
nxo = None
id_to_name = {}

cache_path = NXONTOLOGY_CACHE_PATH
rebuild_cache = False

if cache_path and Path(cache_path).exists() and not rebuild_cache:
    log.info("Loading cached NXOntology from %s ...", cache_path)
    nxo = load_nxontology(cache_path)
    # Check that all our terms are present in the cached graph
    cached_nodes = set(nxo.graph.nodes)
    missing_from_cache = [t for t in all_atoms if t not in cached_nodes]
    if missing_from_cache:
        log.warning(
            "%d terms not found in cached graph — rebuilding. Missing: %s%s",
            len(missing_from_cache),
            missing_from_cache[:5],
            " ..." if len(missing_from_cache) > 5 else "",
        )
        nxo = None  # force rebuild
    else:
        log.info("All %d terms found in cached graph.", len(all_atoms))
        # Build id_to_name from cached graph node attributes
        id_to_name = {n: data.get("name", n) for n, data in nxo.graph.nodes(data=True)}

# --- Build from scratch if no valid cache ---
if nxo is None:
    # Load local OBO ontologies (CL, UBERON)
    merged_graph = load_obo_ontologies(OBO_URLS)
    id_to_name = get_id_to_name(merged_graph)

    # Build initial NXOntology from local data
    nxo_initial = build_nxontology_from_obo(merged_graph)

    # Check which terms need OLS4 augmentation
    local_nodes = set(nxo_initial.graph.nodes)
    needs_ols4 = [t for t in all_atoms if t not in local_nodes]
    log.info(
        "%d / %d terms found in local graph, %d need OLS4 lookup.",
        len(all_atoms) - len(needs_ols4),
        len(all_atoms),
        len(needs_ols4),
    )

    # Augment with OLS4 if needed
    if needs_ols4:
        ols4 = OLS4Client()
        nxo = augment_nxontology_with_ols4(
            nxo_initial.graph,
            all_atoms,
            ols4,
        )
    else:
        nxo = nxo_initial

    # Save to cache
    if cache_path:
        save_nxontology(nxo, cache_path)

In [ ]:
# --- Compute similarity (compound-aware) ---
pairwise_df = compound_pairwise_similarity(nxo, raw_epiatlas, raw_encode)  # type: ignore

# --- Add human-readable names ---
# For compound terms, show names of the best-matching atoms
pairwise_df["term_a_name"] = pairwise_df["best_atom_a"].map(
    lambda x: id_to_name.get(x, x) if x else None
)
pairwise_df["term_b_name"] = pairwise_df["best_atom_b"].map(
    lambda x: id_to_name.get(x, x) if x else None
)

# --- Classify and summarize ---
best_df = summarize_best_matches(pairwise_df)
best_classified = classify_overlap(best_df)

print_summary(best_classified)

# --- Save outputs ---
logdir = project_path / "tables" / "biospecimen_comparison"
output_prefix = "biospecimen_comparison"
pairwise_path = f"{output_prefix}_pairwise.csv"
best_path = f"{output_prefix}_best_matches.csv"

pairwise_df.to_csv(logdir / pairwise_path, index=False)
best_classified.to_csv(logdir / best_path, index=False)
log.info("Saved pairwise similarities to %s", pairwise_path)
log.info("Saved best matches to %s", best_path)

In [ ]:
# --- Within-cohort cell-type similarity matrix (for CV fold creation) ---
# Square term x term Lin-similarity matrix over the EpiATLAS cohort's own
# biospecimen curies (not the EpiATLAS<->ENCODE comparison above). Consumed by
# epiclass.utils.create_similar_celltype_folds to build similarity-stratified
# cross-validation folds. Reuses the same compound-aware similarity.
cohort_sim = compound_pairwise_similarity(nxo, raw_epiatlas, raw_epiatlas)  # type: ignore
celltype_matrix = cohort_sim.pivot(
    index="term_a", columns="term_b", values="lin_similarity"
)

matrix_path = logdir / "epiatlas_celltype_similarity_matrix.csv"
celltype_matrix.to_csv(matrix_path)
log.info(
    "Saved %d x %d cell-type similarity matrix to %s",
    *celltype_matrix.shape,
    matrix_path,
)

In [ ]:
def weighted_overlap_summary(
    best_matches_df: pd.DataFrame,
    df_encode: pd.DataFrame,
    ontology_col: str = BIOSPECIMEN_ID_COL,
) -> pd.DataFrame:
    """
    Weight overlap tiers by the number of ENCODE samples per term.

    Parameters
    ----------
    best_matches_df : pd.DataFrame
        Output of summarize_best_matches + classify_overlap.
        Has 'term_b' (ENCODE term), 'lin_similarity', 'overlap_tier'.
    df_encode : pd.DataFrame
        Original ENCODE DataFrame with one row per sample.
    ontology_col : str
        Column in df_encode containing ontology IDs.

    Returns
    -------
    pd.DataFrame
        Summary with columns: overlap_tier, n_terms, n_samples, pct_samples.
    """
    # Count how many samples use each ENCODE term
    sample_counts = (
        df_encode[ontology_col]
        .value_counts()
        .rename("n_samples")
        .reset_index()
        .rename(columns={ontology_col: "term_b"})
    )

    merged = best_matches_df.merge(sample_counts, on="term_b", how="left")
    merged["n_samples"] = merged["n_samples"].fillna(0).astype(int)

    # Summarize by tier
    summary = (
        merged.groupby("overlap_tier")
        .agg(
            n_terms=("term_b", "count"),
            n_samples=("n_samples", "sum"),
        )
        .reset_index()
    )
    total_samples = summary["n_samples"].sum()
    summary["pct_samples"] = 100 * summary["n_samples"] / total_samples

    # Sort by tier order
    tier_order = ["exact", "high", "moderate", "low", "distinct", "unmapped"]
    summary["_order"] = summary["overlap_tier"].map(
        {t: i for i, t in enumerate(tier_order)}
    )
    summary = summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

    return summary

In [ ]:
summary = weighted_overlap_summary(best_classified, encode_df)
display(summary)
summary.to_csv(logdir / "overlap_summary_weighted.csv", index=False)